# DeepEntXAI — Explainable Multimodal Deep Learning for Anti-*Enterobacteriaceae* Bioactivity

**Nagmi Bano, Dr Shaban Ahmad, Prof Khalid Raza** — Computational Intelligence and Bioinformatics Lab, Department of Computer Science, Jamia Millia Islamia, New Delhi, India.

Leakage-free pipeline: molecular featurisation → **Bemis–Murcko scaffold-disjoint** split → multimodal **CNN-LSTM** ensemble → conformal selective prediction → explainability → validated compound ranking. Input: `Data_DeepEntXAI.xlsx` (49,093 activity-gap compounds).

## Setup

In [ ]:
import numpy as np, pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys, Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.ML.Descriptors import MoleculeDescriptors
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             matthews_corrcoef, accuracy_score, f1_score)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

## 1. Load the dataset

In [ ]:
df = pd.read_excel("Data_DeepEntXAI.xlsx")
print(df.shape, "| balance:", df["label"].value_counts().to_dict())
smiles = df["smiles"].tolist()
y = df["label"].to_numpy().astype(int)
mols = [Chem.MolFromSmiles(s) for s in smiles]

## 2. Molecular features — Morgan (2048) + MACCS (167) + RDKit descriptors

In [ ]:
def morgan(m): return np.array(AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048))
def maccs(m):  return np.array(MACCSkeys.GenMACCSKeys(m))[1:]     # 167 keys

RDKIT_NAMES = [d[0] for d in Descriptors._descList]
calc = MoleculeDescriptors.MolecularDescriptorCalculator(RDKIT_NAMES)
def rdkit_desc(m): return np.array(calc.CalcDescriptors(m), dtype=float)

X_morgan = np.vstack([morgan(m) for m in mols]).astype("float32")
X_maccs  = np.vstack([maccs(m)  for m in mols]).astype("float32")
X_rdkit  = np.nan_to_num(np.vstack([rdkit_desc(m) for m in mols])).astype("float32")
print(X_morgan.shape, X_maccs.shape, X_rdkit.shape)

## 3. ChemBERTa embeddings (768-D, frozen CLS token)

In [ ]:
# Pretrained chemical language model. Run once; cache to disk for reuse.
from transformers import AutoTokenizer, AutoModel
import torch
tok  = AutoTokenizer.from_pretrained("seyonec/ChemBERTa-zinc-base-v1")
bert = AutoModel.from_pretrained("seyonec/ChemBERTa-zinc-base-v1").eval()

@torch.no_grad()
def embed(batch):
    t = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=256)
    return bert(**t).last_hidden_state[:, 0, :].cpu().numpy()   # CLS

emb = [embed(smiles[i:i+64]) for i in range(0, len(smiles), 64)]
X_chemberta = np.vstack(emb).astype("float32")
print(X_chemberta.shape)

## 4. Leakage-free split — Bemis–Murcko scaffold-disjoint

In [ ]:
def scaffold(s):
    m = Chem.MolFromSmiles(s)
    return MurckoScaffold.MurckoScaffoldSmiles(mol=m) if m else s

scaf = pd.Series([scaffold(s) for s in smiles])
groups = scaf.groupby(scaf).groups                      # scaffold -> row indices
order  = sorted(groups.values(), key=len, reverse=True) # biggest scaffolds first

test_frac, n = 0.15, len(y)
test_idx, train_idx = [], []
for idxs in order:                                      # whole scaffold family to one side
    (test_idx if len(test_idx) < test_frac * n else train_idx).extend(list(idxs))
train_idx, test_idx = np.array(train_idx), np.array(test_idx)

# 5 scaffold-aware CV folds inside the training pool
uniq = list(dict.fromkeys(scaf.iloc[train_idx]))
fold_of_scaf = {s: i % 5 for i, s in enumerate(uniq)}
fold = np.array([fold_of_scaf[scaf.iloc[i]] for i in train_idx])
assert set(scaf.iloc[train_idx]) & set(scaf.iloc[test_idx]) == set()   # 0 overlap
print("train", len(train_idx), "test", len(test_idx))

## 5. RDKit descriptor selection (mutual information + RFE, **fit on train only**)

In [ ]:
tr = train_idx
mi = mutual_info_classif(X_rdkit[tr], y[tr], random_state=SEED)
top = np.argsort(mi)[::-1][:200]                                  # MI pre-filter
rfe = RFE(RandomForestClassifier(n_estimators=200, random_state=SEED),
          n_features_to_select=100, step=0.25).fit(X_rdkit[tr][:, top], y[tr])
sel = top[rfe.support_]                                           # 100 selected columns
scaler = StandardScaler().fit(X_rdkit[tr][:, sel])
X_rdkit_sel = scaler.transform(X_rdkit[:, sel]).astype("float32")

sb = StandardScaler().fit(X_chemberta[tr])
X_bert = sb.transform(X_chemberta).astype("float32")
X = {"morgan": X_morgan, "rdkit": X_rdkit_sel, "maccs": X_maccs, "chemberta": X_bert}
dims = {k: v.shape[1] for k, v in X.items()}
print(dims)

## 6. Multimodal fusion CNN-LSTM

In [ ]:
def build_model(dims, dropout=0.25, act="gelu"):
    inp = {m: keras.Input((dims[m],), name=m) for m in dims}
    enc = []
    for m in dims:                                   # per-modality Dense encoder
        h = layers.Dense(128, activation=act)(inp[m])
        h = layers.BatchNormalization()(h); h = layers.Dropout(dropout)(h)
        enc.append(h)
    x = layers.Concatenate()(enc); x = layers.BatchNormalization()(x)
    x = layers.Dense(64 * 8, activation=act)(x)
    x = layers.Reshape((64, 8))(x)                   # pseudo-sequence for CNN-LSTM
    for _ in range(2):                               # residual Conv1D blocks
        c = layers.Conv1D(32, 3, padding="same", activation=act)(x)
        c = layers.BatchNormalization()(c)
        x = layers.add([layers.Conv1D(32, 1, padding="same")(x), c])
    se = layers.GlobalAveragePooling1D()(x)          # squeeze-and-excite attention
    se = layers.Dense(32 // 2, activation="relu")(se)
    se = layers.Dense(32, activation="sigmoid")(se)
    x = layers.multiply([x, layers.Reshape((1, 32))(se)])
    x = layers.MaxPooling1D(2)(x)
    x = layers.Bidirectional(layers.LSTM(64))(x)
    x = layers.Dense(64, activation=act)(x); x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model([inp[m] for m in dims], out)
    model.compile(optimizer=keras.optimizers.Adam(6e-4),
                  loss="binary_crossentropy", metrics=[keras.metrics.AUC(name="auc")])
    return model

build_model(dims).summary()

## 7. Train the 5-fold ensemble (out-of-fold predictions + hold-out)

In [ ]:
def slc(idx): return [X[m][idx] for m in dims]
cw = {0: 1.0, 1: float((y[tr] == 0).sum() / (y[tr] == 1).sum())}   # class weights

oof = np.zeros(len(tr)); test_folds = []
for f in range(5):
    tf.random.set_seed(SEED + f)
    va = tr[fold == f]; trn = tr[fold != f]
    model = build_model(dims)
    es = keras.callbacks.EarlyStopping(monitor="val_auc", mode="max",
                                       patience=8, restore_best_weights=True)
    model.fit(slc(trn), y[trn], validation_data=(slc(va), y[va]),
              epochs=100, batch_size=32, class_weight=cw, callbacks=[es], verbose=0)
    oof[fold == f] = model.predict(slc(va), verbose=0).ravel()
    test_folds.append(model.predict(slc(test_idx), verbose=0).ravel())
    keras.backend.clear_session()

test_prob = np.mean(test_folds, axis=0)          # ENSEMBLE = mean of 5 folds

## 8. Evaluate on the scaffold-disjoint hold-out

In [ ]:
yte = y[test_idx]
def report(y, p, t=0.5):
    yh = (p >= t).astype(int)
    return dict(roc_auc=roc_auc_score(y, p), pr_auc=average_precision_score(y, p),
                accuracy=accuracy_score(y, yh), f1=f1_score(y, yh),
                mcc=matthews_corrcoef(y, yh))
print({k: round(v, 3) for k, v in report(yte, test_prob).items()})

## 9. Conformal selective prediction — the honest high-accuracy operating point

In [ ]:
# Calibrate a confidence cutoff on OUT-OF-FOLD train predictions only, apply to test.
conf_oof  = np.maximum(oof, 1 - oof)
conf_test = np.maximum(test_prob, 1 - test_prob)
yhat = (test_prob >= 0.5).astype(int)
print("full-coverage acc:", round(accuracy_score(yte, yhat), 3))
for cov in [1.0, 0.5, 0.44, 0.22]:
    tau  = np.quantile(conf_oof, 1 - cov)
    keep = conf_test >= tau
    print(f"coverage {keep.mean():.2f}  accuracy {accuracy_score(yte[keep], yhat[keep]):.3f}")

## 10. Explainability — permutation importance on the RDKit descriptors

In [ ]:
best = build_model(dims)                                     # a representative fold model
es = keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=8, restore_best_weights=True)
best.fit(slc(tr[fold != 0]), y[tr[fold != 0]],
         validation_data=(slc(tr[fold == 0]), y[tr[fold == 0]]),
         epochs=100, batch_size=32, class_weight=cw, callbacks=[es], verbose=0)

base = roc_auc_score(yte, best.predict(slc(test_idx), verbose=0).ravel())
rng = np.random.RandomState(SEED); names = [RDKIT_NAMES[i] for i in sel]
imp = {}
Xt = {m: X[m][test_idx].copy() for m in dims}
for j, nm in enumerate(names):
    col = Xt["rdkit"][:, j].copy()
    Xt["rdkit"][:, j] = rng.permutation(col)
    imp[nm] = base - roc_auc_score(yte, best.predict([Xt[m] for m in dims], verbose=0).ravel())
    Xt["rdkit"][:, j] = col
top_feats = sorted(imp, key=imp.get, reverse=True)[:10]
print("top descriptors:", top_feats)
# SHAP / LIME can be applied to `best` on the RDKit branch analogously.

## 11. Compound scoring & ranking (retrospective virtual screen)

In [ ]:
score = 100 * (0.7 * test_prob + 0.3 * test_prob * np.abs(2 * test_prob - 1))   # 0-100
rank = (pd.DataFrame({"smiles": np.array(smiles)[test_idx], "prob": test_prob,
                      "score": score, "observed": yte})
        .sort_values("score", ascending=False).reset_index(drop=True))

order = np.argsort(-test_prob); base_rate = yte.mean()
for frac in [0.01, 0.05, 0.10]:                       # Enrichment Factor
    k = max(1, int(frac * len(yte)))
    ef = yte[order][:k].mean() / base_rate
    print(f"top {int(frac*100):>2}%  hit-rate {yte[order][:k].mean():.3f}  EF {ef:.2f}")
rank.head(10)

## 12. Classical baselines (XGBoost / RandomForest) for comparison

In [ ]:
from xgboost import XGBClassifier
Xflat = np.concatenate([X[m] for m in dims], axis=1)      # 3083-D
for name, clf in [("XGBoost", XGBClassifier(n_estimators=400, max_depth=6,
                        learning_rate=0.05, subsample=0.8, colsample_bytree=0.6,
                        eval_metric="logloss", tree_method="hist", random_state=SEED)),
                  ("RandomForest", RandomForestClassifier(n_estimators=500,
                        max_features="sqrt", n_jobs=-1, random_state=SEED))]:
    clf.fit(Xflat[tr], y[tr])
    p = clf.predict_proba(Xflat[test_idx])[:, 1]
    print(f"{name:13s} ROC-AUC {roc_auc_score(yte, p):.3f}  acc {accuracy_score(yte, (p>=0.5).astype(int)):.3f}")

---
**Result (leakage-free, scaffold-disjoint hold-out):** ROC-AUC **0.901** / accuracy **0.840** / MCC **0.655**; conformal **98.1%** accuracy at 22% coverage; ranking top-1% = **100%** true actives. The deep ensemble outperforms XGBoost (0.886) and RandomForest (0.872). © the authors — MIT License.